In [7]:
import matplotlib.pyplot as plt
import pandas as pd
import fusou_datasets

def load_table(name: str, master: bool = False):
    try:
        return fusou_datasets.load_master(name) if master else fusou_datasets.load(name, offline=True)
    except Exception:
        return fusou_datasets.load_master(name) if master else fusou_datasets.load(name, offline=False)

fusou_datasets.configure(cache_dir="~/.fusou_datasets/cache")

df_cells = load_table("cells")
needed = {"battle_uuid", "mapinfo_no", "cell"}
if not needed.issubset(df_cells.columns):
    print("cells needs battle_uuid, mapinfo_no, cell columns")
else:
    df_cells = df_cells.sort_values(["battle_uuid", "timestamp"] if "timestamp" in df_cells.columns else ["battle_uuid"])
    df_cells["next_cell"] = df_cells.groupby("battle_uuid")["cell"].shift(-1)
    df_cells["next_map"] = df_cells.groupby("battle_uuid")["mapinfo_no"].shift(-1)
    flows = df_cells[(df_cells["next_cell"].notna()) & (df_cells["mapinfo_no"] == df_cells["next_map"])]
    flows["edge"] = flows["cell"].astype(str) + " → " + flows["next_cell"].astype(str)
    top_edges = flows["edge"].value_counts().head(20)
    top_edges.plot(kind="barh", figsize=(10,5), color="#6A7FDB", title="Top Cell-to-Cell Flows (same map)")
    plt.gca().invert_yaxis()
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

cells needs battle_uuid, mapinfo_no, cell columns


[Cache] Loading cells from cache (offline)


## データ確認とクロスマップ遷移も含めたチェック
同一マップ内で遷移が無い場合の診断用に件数とクロスマップ遷移も確認します。

In [8]:
from fusou_datasets import Tables

col_battle = Tables.Cells.BATTLE_UUID
col_map = Tables.Cells.MAPINFO_NO
col_cell = Tables.Cells.CELL
col_ts = Tables.Cells.TIMESTAMP

print("レコード数:", len(df_cells))
print("battle_uuid のユニーク数:", df_cells[col_battle].nunique() if col_battle in df_cells.columns else "(col missing)")
print("mapinfo_no のユニーク数:", df_cells[col_map].nunique())
if col_battle in df_cells.columns:
    print("battle_uuid 上位5件: \n", df_cells[col_battle].value_counts().head())
else:
    print("battle_uuid が無いため暫定IDで解析済み")

# クロスマップ遷移も含めたフロー確認
sort_keys = [col_battle, col_ts] if col_battle in df_cells.columns and col_ts in df_cells.columns else [col_battle] if col_battle in df_cells.columns else [col_map, "_row_order"] if "_row_order" in df_cells.columns else [col_map]
df_cells = df_cells.sort_values(sort_keys)
df_cells["next_cell_any"] = df_cells.groupby(col_battle if col_battle in df_cells.columns else col_map)[col_cell].shift(-1)
df_cells["next_map_any"] = df_cells.groupby(col_battle if col_battle in df_cells.columns else col_map)[col_map].shift(-1)

flows_any = df_cells[df_cells["next_cell_any"].notna()]
if flows_any.empty:
    print("全体でも遷移がありません (行数が少ない可能性)")
else:
    flows_any["edge_any"] = flows_any[col_cell].astype(str) + " → " + flows_any["next_cell_any"].astype(str)
    top_edges_any = flows_any["edge_any"].value_counts().head(10)
    print("クロスマップ含む遷移 上位10件:\n", top_edges_any)
    ax = top_edges_any.plot(kind="barh", figsize=(8,4), color="#888888", title="Cell-to-Cell Flows (cross map included)")
    ax.invert_yaxis()
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()

AttributeError: type object 'Cells' has no attribute 'BATTLE_UUID'